# Module 02 — Optimizer Rules & Join Strategies

Column pruning, filter pushdown, equi vs inequi joins, and how to read join nodes in `explain`.

In [2]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs
from shared.demo_data import user_post_activity

from pyspark.sql import functions as F
spark = begin_module('module-02-plan-joins')
ui_links()

{'master_ui': 'http://localhost:8080',
 'app_ui': 'http://localhost:4040',
 'worker_1_ui': 'http://localhost:8081',
 'worker_2_ui': 'http://localhost:8082',
 'app_ui_internal': 'http://jupyter-lab:4040'}

In [3]:
trips = spark.read.parquet(*config.yellow_taxi_paths())
zones = spark.read.option('header', True).csv(config.zones_path())
trips_small = trips.select(
    'VendorID', 'PULocationID', 'DOLocationID', 'total_amount', 'tpep_pickup_datetime'
)

## Column pruning

In [4]:
wide = trips  # many columns
narrow = trips.select('VendorID', 'total_amount')
narrow.explain('formatted')  # fewer Parquet columns read
wide.filter('total_amount > 80').select('VendorID').explain('formatted')

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [VendorID#0, total_amount#16]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
ReadSchema: struct<VendorID:int,total_amount:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [VendorID#0, total_amount#16]


== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [VendorID#0, total_amount#16]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [IsNotNull(total_amount), GreaterThan(total_amount,80.0)]
ReadSchema: struct<VendorID:int,total_amount:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [VendorID#0, total_amount#16]

(3) Filter [codegen id : 1]
Input [2]: [VendorID#0, total_amount#16]
Condition : (isnotnull(total_amount#16) AND (total_amount#16 > 80.0))



## Filter pushdown

In [5]:
trips_small.createOrReplaceTempView("trips_small")

q_cte = spark.sql("""
WITH picked AS (
    SELECT
        VendorID,
        PULocationID,
        total_amount,
        tpep_pickup_datetime
    FROM trips_small
)
SELECT VendorID, PULocationID, total_amount
FROM picked
WHERE total_amount > 100 AND VendorID = 2
""")
q_cte.explain("formatted")

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [VendorID#0, PULocationID#7, total_amount#16]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [IsNotNull(total_amount), IsNotNull(VendorID), GreaterThan(total_amount,100.0), EqualTo(VendorID,2)]
ReadSchema: struct<VendorID:int,PULocationID:int,total_amount:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [VendorID#0, PULocationID#7, total_amount#16]

(3) Filter [codegen id : 1]
Input [3]: [VendorID#0, PULocationID#7, total_amount#16]
Condition : (((isnotnull(total_amount#16) AND isnotnull(VendorID#0)) AND (total_amount#16 > 100.0)) AND (VendorID#0 = 2))




## Equi-join baseline (sort-merge at scale)

In [6]:
reset_spark_configs(spark, {
    'spark.sql.autoBroadcastJoinThreshold': '-1',
    'spark.sql.join.preferSortMergeJoin': 'true',
})
equi = trips_small.alias('t').join(
    trips_small.alias('s'),
    (F.col('t.VendorID') == F.col('s.VendorID'))
    & (F.col('t.PULocationID') == F.col('s.PULocationID')),
)
equi.select('t.VendorID', 't.PULocationID').limit(5).explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (12)
+- CollectLimit (11)
   +- Project (10)
      +- SortMergeJoin Inner (9)
         :- Sort (4)
         :  +- Exchange (3)
         :     +- Filter (2)
         :        +- Scan parquet  (1)
         +- Sort (8)
            +- Exchange (7)
               +- Filter (6)
                  +- Scan parquet  (5)


(1) Scan parquet 
Output [2]: [VendorID#0, PULocationID#7]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [IsNotNull(VendorID), IsNotNull(PULocationID)]
ReadSchema: struct<VendorID:int,PULocationID:int>

(2) Filter
Input [2]: [VendorID#0, PULocationID#7]
Condition : (isnotnull(VendorID#0) AND isnotnull(PULocationID#7))

(3) Exchange
Input [2]: [VendorID#0, PULocationID#7]
Arguments: hashpartitioning(VendorID#0, PULocationID#7, 200), ENSURE_REQUIREMENTS, [plan_id=88]

(4) Sort
Input [2]: [VendorID#0, PULocationID#7]
Arguments: [VendorID#0 ASC NULLS FIRST, P

## Broadcast hash join (small dimension)

In [7]:
z = zones.withColumnRenamed('LocationID', 'PULocationID')
reset_spark_configs(spark, {'spark.sql.autoBroadcastJoinThreshold': '52428800'})
bj = trips_small.join(z, 'PULocationID', 'inner')
bj.explain('formatted')  # BroadcastExchange / BroadcastHashJoin

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- BroadcastHashJoin Inner BuildRight (7)
      :- Filter (2)
      :  +- Scan parquet  (1)
      +- BroadcastExchange (6)
         +- Project (5)
            +- Filter (4)
               +- Scan csv  (3)


(1) Scan parquet 
Output [5]: [VendorID#0, tpep_pickup_datetime#1, PULocationID#7, DOLocationID#8, total_amount#16]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [IsNotNull(PULocationID)]
ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,PULocationID:int,DOLocationID:int,total_amount:double>

(2) Filter
Input [5]: [VendorID#0, tpep_pickup_datetime#1, PULocationID#7, DOLocationID#8, total_amount#16]
Condition : isnotnull(PULocationID#7)

(3) Scan csv 
Output [4]: [LocationID#36, Borough#37, Zone#38, service_zone#39]
Batched: false
Location: InMemoryFileIndex [file:/workspace/data/cache/taxi_zone_lookup.csv]
PushedFilters: 

## Shuffle hash join (disable sort-merge preference)

In [8]:
reset_spark_configs(spark, {
    'spark.sql.autoBroadcastJoinThreshold': '-1',
    'spark.sql.join.preferSortMergeJoin': 'false',
})
big = trips_small.repartition(32, 'VendorID')
shj = big.alias('a').join(big.hint('shuffle_hash').alias('b'), 'VendorID', 'inner')
shj.limit(1).explain('formatted')  # ShuffledHashJoin when chosen

== Physical Plan ==
AdaptiveSparkPlan (12)
+- CollectLimit (11)
   +- Project (10)
      +- ShuffledHashJoin Inner BuildRight (9)
         :- Exchange (4)
         :  +- Project (3)
         :     +- Filter (2)
         :        +- Scan parquet  (1)
         +- Exchange (8)
            +- Project (7)
               +- Filter (6)
                  +- Scan parquet  (5)


(1) Scan parquet 
Output [5]: [VendorID#0, tpep_pickup_datetime#1, PULocationID#7, DOLocationID#8, total_amount#16]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [IsNotNull(VendorID)]
ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,PULocationID:int,DOLocationID:int,total_amount:double>

(2) Filter
Input [5]: [VendorID#0, tpep_pickup_datetime#1, PULocationID#7, DOLocationID#8, total_amount#16]
Condition : isnotnull(VendorID#0)

(3) Project
Output [5]: [VendorID#0, PULocationID#7, DOLocationID#8, total_amount#16, tpep_

## Sort-merge join (default for large equi)

In [9]:
reset_spark_configs(spark, {
    'spark.sql.autoBroadcastJoinThreshold': '-1',
    'spark.sql.join.preferSortMergeJoin': 'true',
})
smj = big.alias('a').join(big.alias('b'), 'PULocationID', 'inner')
smj.limit(1).explain('formatted')  # SortMergeJoin

== Physical Plan ==
AdaptiveSparkPlan (16)
+- CollectLimit (15)
   +- Project (14)
      +- SortMergeJoin Inner (13)
         :- Sort (6)
         :  +- Exchange (5)
         :     +- Exchange (4)
         :        +- Project (3)
         :           +- Filter (2)
         :              +- Scan parquet  (1)
         +- Sort (12)
            +- Exchange (11)
               +- Exchange (10)
                  +- Project (9)
                     +- Filter (8)
                        +- Scan parquet  (7)


(1) Scan parquet 
Output [5]: [VendorID#0, tpep_pickup_datetime#1, PULocationID#7, DOLocationID#8, total_amount#16]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [IsNotNull(PULocationID)]
ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,PULocationID:int,DOLocationID:int,total_amount:double>

(2) Filter
Input [5]: [VendorID#0, tpep_pickup_datetime#1, PULocationID#7, DOLocationID#8, tot

## Cartesian / broadcast nested loop (dangerous at scale)

In [10]:
tiny_a = trips_small.select('VendorID').distinct().limit(5)
tiny_b = trips_small.select('PULocationID').distinct().limit(5)
cross = tiny_a.crossJoin(tiny_b)
cross.explain('formatted')
spark.conf.set('spark.sql.crossJoin.enabled', 'true')

== Physical Plan ==
AdaptiveSparkPlan (16)
+- CartesianProduct Inner (15)
   :- GlobalLimit (7)
   :  +- Exchange (6)
   :     +- LocalLimit (5)
   :        +- HashAggregate (4)
   :           +- Exchange (3)
   :              +- HashAggregate (2)
   :                 +- Scan parquet  (1)
   +- GlobalLimit (14)
      +- Exchange (13)
         +- LocalLimit (12)
            +- HashAggregate (11)
               +- Exchange (10)
                  +- HashAggregate (9)
                     +- Scan parquet  (8)


(1) Scan parquet 
Output [1]: [VendorID#0]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
ReadSchema: struct<VendorID:int>

(2) HashAggregate
Input [1]: [VendorID#0]
Keys [1]: [VendorID#0]
Functions: []
Aggregate Attributes: []
Results [1]: [VendorID#0]

(3) Exchange
Input [1]: [VendorID#0]
Arguments: hashpartitioning(VendorID#0, 200), ENSURE_REQUIREMENTS, [plan_id=250]

(4) HashAggregate
Input [1]: [VendorID#0]


In [11]:
from pyspark.sql.functions import broadcast

inequi_loop = tiny_a.alias('a').join(
    broadcast(tiny_b).alias('b'),
    F.col('a.VendorID') < F.col('b.PULocationID'),
    how='inner',
)
inequi_loop.explain('formatted')  # often BroadcastNestedLoopJoin or Cartesian

== Physical Plan ==
AdaptiveSparkPlan (19)
+- BroadcastNestedLoopJoin Inner BuildRight (18)
   :- Filter (8)
   :  +- GlobalLimit (7)
   :     +- Exchange (6)
   :        +- LocalLimit (5)
   :           +- HashAggregate (4)
   :              +- Exchange (3)
   :                 +- HashAggregate (2)
   :                    +- Scan parquet  (1)
   +- BroadcastExchange (17)
      +- Filter (16)
         +- GlobalLimit (15)
            +- Exchange (14)
               +- LocalLimit (13)
                  +- HashAggregate (12)
                     +- Exchange (11)
                        +- HashAggregate (10)
                           +- Scan parquet  (9)


(1) Scan parquet 
Output [1]: [VendorID#0]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
ReadSchema: struct<VendorID:int>

(2) HashAggregate
Input [1]: [VendorID#0]
Keys [1]: [VendorID#0]
Functions: []
Aggregate Attributes: []
Results [1]: [VendorID#0]

(3) Exchange

## Inequi join challenge — decompose to equi + filter

Predicate: match trips whose `total_amount` is within $5 (range / inequi).
Instead of `abs(t-s) < 5` in the join, **bucket** amounts and join on equi keys, then filter.

In [12]:
from pyspark.sql import functions as F

a = trips_small.select("VendorID", "PULocationID").limit(10_000)
b = a.select(
    F.col("VendorID").alias("v_id"),
    F.col("PULocationID").alias("pu_id"),
)

spark.conf.set("spark.sql.adaptive.enabled", "false")

In [13]:
bad = a.join(
    b,
    (F.col("VendorID") == F.col("v_id"))
    | (F.col("PULocationID") == F.col("pu_id")),
    "inner",
)

bad.explain("formatted")

== Physical Plan ==
CartesianProduct Inner (12)
:- * GlobalLimit (5)
:  +- Exchange (4)
:     +- * LocalLimit (3)
:        +- * ColumnarToRow (2)
:           +- Scan parquet  (1)
+- * GlobalLimit (11)
   +- Exchange (10)
      +- * LocalLimit (9)
         +- * Project (8)
            +- * ColumnarToRow (7)
               +- Scan parquet  (6)


(1) Scan parquet 
Output [2]: [VendorID#0, PULocationID#7]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
ReadSchema: struct<VendorID:int,PULocationID:int>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [VendorID#0, PULocationID#7]

(3) LocalLimit [codegen id : 1]
Input [2]: [VendorID#0, PULocationID#7]
Arguments: 10000

(4) Exchange
Input [2]: [VendorID#0, PULocationID#7]
Arguments: SinglePartition, ENSURE_REQUIREMENTS, [plan_id=378]

(5) GlobalLimit [codegen id : 2]
Input [2]: [VendorID#0, PULocationID#7]
Arguments: 10000, 0

(6) Scan parquet 
Output [2]: [VendorID#145, PULo

In [14]:
l = a.alias("l")
bv = b.alias("bv")
bz = b.alias("bz")

good = (
    l.join(bv, F.col("l.VendorID") == F.col("bv.v_id"), "left")
    .join(bz, F.col("l.PULocationID") == F.col("bz.pu_id"), "left")
    .filter(F.col("bv.v_id").isNotNull() | F.col("bz.pu_id").isNotNull())
    .select(
        F.col("l.VendorID"),
        F.col("l.PULocationID"),
        F.coalesce(F.col("bv.v_id"), F.col("bz.v_id")).alias("v_id"),
        F.coalesce(F.col("bz.pu_id"), F.col("bv.pu_id")).alias("pu_id"),
    )
)
good.explain("formatted")

== Physical Plan ==
* Project (25)
+- * Filter (24)
   +- * SortMergeJoin LeftOuter (23)
      :- * Sort (17)
      :  +- Exchange (16)
      :     +- * SortMergeJoin LeftOuter (15)
      :        :- * Sort (6)
      :        :  +- * GlobalLimit (5)
      :        :     +- Exchange (4)
      :        :        +- * LocalLimit (3)
      :        :           +- * ColumnarToRow (2)
      :        :              +- Scan parquet  (1)
      :        +- * Sort (14)
      :           +- * Filter (13)
      :              +- * GlobalLimit (12)
      :                 +- Exchange (11)
      :                    +- * LocalLimit (10)
      :                       +- * Project (9)
      :                          +- * ColumnarToRow (8)
      :                             +- Scan parquet  (7)
      +- * Sort (22)
         +- Exchange (21)
            +- * Filter (20)
               +- * GlobalLimit (19)
                  +- ReusedExchange (18)


(1) Scan parquet 
Output [2]: [VendorID#0, PULocationID